# speech playground

record your voice -> transcribe (stt), design a voice -> generate speech (tts).
reads `RUNPOD_API_KEY`, `STT_ENDPOINT_ID`, `TTS_ENDPOINT_ID` from `.env`.

In [1]:
import base64
import io
import os
from pathlib import Path

import requests
from IPython.display import Audio, display

env_path = next(
    (p / ".env" for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()), None
)
for line in env_path.read_text().splitlines():
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())

API = "https://api.runpod.ai/v2"
HEADERS = {"Authorization": f"Bearer {os.environ['RUNPOD_API_KEY']}"}


def call(endpoint_id, input_payload):
    resp = requests.post(
        f"{API}/{endpoint_id}/runsync",
        headers=HEADERS,
        json={"input": input_payload},
        timeout=600,
    )
    resp.raise_for_status()
    output = resp.json().get("output") or {}
    if output.get("error"):
        raise RuntimeError(output["error"])
    return output


def transcribe(audio_bytes, language=None):
    payload = {"audio": base64.b64encode(audio_bytes).decode()}
    if language:
        payload["language"] = language
    return call(os.environ["STT_ENDPOINT_ID"], payload)


def speak(text, instruct="", language="Auto", **sampling):
    output = call(
        os.environ["TTS_ENDPOINT_ID"],
        {"text": text, "instruct": instruct, "language": language, **sampling},
    )
    audio = base64.b64decode(output["audio"])
    return Audio(audio, rate=output["sample_rate"])


def chat(messages, max_tokens=256, **sampling):
    output = call(
        os.environ["LLM_ENDPOINT_ID"],
        {"messages": messages, "max_tokens": max_tokens, **sampling},
    )
    print(output["text"])
    return output


def record(seconds=5, rate=16000):
    import sounddevice as sd
    import soundfile as sf

    print(f"recording {seconds}s... speak now")
    audio = sd.rec(int(seconds * rate), samplerate=rate, channels=1, dtype="int16")
    sd.wait()
    buffer = io.BytesIO()
    sf.write(buffer, audio, rate, format="WAV")
    return buffer.getvalue()


print("ready")

ready


## 1. record your voice and transcribe it

In [2]:
my_voice = record(seconds=5)
display(Audio(my_voice, rate=16000))
transcribe(my_voice)

recording 5s... speak now


{'language': '', 'text': ''}

## 2. design a voice and listen

In [3]:
speak(
    "Hi! This voice was designed on demand, just for you.",
    instruct="A cute child's voice, around 8 years old, speaking with a "
    "slightly childish tone, suitable for animation character voice-overs.",
    language="English",
)

In [4]:
# another voice - same text, different character
speak(
    "Hi! This voice was designed on demand, just for you.",
    instruct="elderly man, gravelly voice, slow and thoughtful",
    language="English",
    temperature=0.8,
)

## 3. voice design: how good is it?

a mini evaluation suite - every voice is its own cell, run the ones you want
and listen. each cell fires exactly one tts job.

### speaker profiles - same line, different age / gender / timbre

In [ ]:
speak(
    'Hi! This voice was designed on demand, just for you.',
    instruct="a cute child's voice, around 8 years old, playful",
    language='English',
)

In [ ]:
speak(
    'Hi! This voice was designed on demand, just for you.',
    instruct='young woman in her 20s, bright and energetic',
    language='English',
)

In [ ]:
speak(
    'Hi! This voice was designed on demand, just for you.',
    instruct='middle-aged man, warm baritone, calm and steady',
    language='English',
)

In [ ]:
speak(
    'Hi! This voice was designed on demand, just for you.',
    instruct='elderly woman, soft and slightly raspy, unhurried',
    language='English',
)

### emotions - same sentence, different feelings

In [ ]:
speak(
    "I can't believe we actually won the championship!",
    instruct='young woman, whispered excitement, barely containing joy',
    language='English',
)

In [ ]:
speak(
    "I can't believe we actually won the championship!",
    instruct='stadium announcer, shouting with pure joy, breathless',
    language='English',
)

In [ ]:
speak(
    "I can't believe we actually won the championship!",
    instruct='young man, voice breaking, fighting back tears',
    language='English',
)

In [ ]:
speak(
    "I can't believe we actually won the championship!",
    instruct='middle-aged man, completely deadpan and unimpressed',
    language='English',
)

### accents - same tongue twister, different english accents

In [ ]:
speak(
    'Around the rugged rocks the ragged rascal ran.',
    instruct='British RP accent, crisp and precise',
    language='English',
)

In [ ]:
speak(
    'Around the rugged rocks the ragged rascal ran.',
    instruct='American southern drawl, relaxed and friendly',
    language='English',
)

In [ ]:
speak(
    'Around the rugged rocks the ragged rascal ran.',
    instruct='Indian English accent, clear and warm',
    language='English',
)

In [ ]:
speak(
    'Around the rugged rocks the ragged rascal ran.',
    instruct='Scottish accent, lilting',
    language='English',
)

### speaking styles - character performance

In [ ]:
speak(
    "Welcome back to the show! Tonight's guest changed everything.",
    instruct='energetic podcast host, quick pace, conversational',
    language='English',
)

In [ ]:
speak(
    'In a world where silence ruled the skies, one voice dared to speak.',
    instruct='dramatic movie trailer voice, deep, slow epic build',
    language='English',
)

In [ ]:
speak(
    "And it's a curling shot from outside the box... GOOOAL! What a strike in the final minute!",
    instruct='football commentator, rapid-fire excitement',
    language='English',
)

In [ ]:
speak(
    'Once upon a time, in a quiet little village at the edge of the forest...',
    instruct='bedtime storyteller, gentle whisper, very slow pace',
    language='English',
)

### languages - non-english fluency with native voice design

In [ ]:
speak(
    '欢迎收听今天的天气预报，明天会有小雨，出门记得带伞。',
    instruct='成年女性，新闻主播风格，字正腔圆',
    language='Chinese',
)

In [ ]:
speak(
    "Bonjour ! Aujourd'hui, nous allons parler de l'art de la pâtisserie française.",
    instruct='femme parisienne, chaleureuse et vive, débit modéré',
    language='French',
)

In [ ]:
speak(
    '本日のニュースをお伝えします。今後一週間、晴れが続く見込みです。',
    instruct='日本女性のアナウンサー、落ち着いた感じ',
    language='Japanese',
)

### robustness - numbers, currency, email, dense punctuation

In [ ]:
speak(
    'Your meeting is at 3:45pm on March 7th; the invoice total is $1,299.50, and our support email is help@qwen.ai. Reply before Friday, or call 555-0142.',
    instruct='professional assistant, clear enunciation, medium pace',
    language='English',
)

## 4. chat with qwen3-14b


In [5]:
chat([{"role": "user", "content": "Explain speculative decoding in two sentences."}])


Speculative decoding is a technique used in large language models to improve inference speed by generating multiple potential outputs in parallel and selecting the most likely one. It allows the model to hypothesize future tokens before they are confirmed, reducing the total number of steps needed for generation.


{'completion_tokens': 54,
 'finish_reason': 'stop',
 'prompt_tokens': 20,
 'text': 'Speculative decoding is a technique used in large language models to improve inference speed by generating multiple potential outputs in parallel and selecting the most likely one. It allows the model to hypothesize future tokens before they are confirmed, reducing the total number of steps needed for generation.'}

## 5. full loop: llm writes it, tts speaks it


In [6]:
reply = chat(
    [{"role": "user", "content": "Say something inspiring in one short sentence."}],
    max_tokens=64,
)
speak(reply["text"], instruct="warm confident narrator, medium pace", language="English")


Believe in your potential, and the world will follow your lead.


## 6. streaming endpoints

`qwen3-14b-stream` and `qwen3-asr-1.7b-stream` emit partial output while the job runs
(`/run` submit + `/stream/<job_id>` polling). both sit at `workersMax: 0` to respect the
5-worker quota - to use one, borrow a slot (e.g. drop `qwen3-asr-1.7b` to 1) and set the
stream endpoint's max to 1, then flip back after. reads `LLM_STREAM_ENDPOINT_ID` and
`STT_STREAM_ENDPOINT_ID` from `.env`.

note: during a cold start the queue is silent (image pull + model load, minutes) -
output starts appearing only once the worker is up. chunks arrive in small batches,
one batch per poll (every 0.5s); everything below prints with `flush=True` so jupyter
shows it live.

In [ ]:
import time


def stream(endpoint_id, input_payload, poll_interval=0.5):
    """run a streaming job: yield each partial chunk as it arrives.

    prints nothing itself - callers decide how to display. chunks arrive in
    batches (one batch per poll); during cold start there is nothing to show
    until the worker finishes booting.
    """
    job_id = requests.post(
        f"{API}/{endpoint_id}/run", headers=HEADERS, json={"input": input_payload}, timeout=60
    ).json()["id"]
    while True:
        body = requests.post(
            f"{API}/{endpoint_id}/stream/{job_id}", headers=HEADERS, timeout=60
        ).json()
        for chunk in body.get("stream") or []:
            if isinstance(chunk, dict) and chunk.get("error"):
                raise RuntimeError(chunk["error"])
            yield chunk
        status = requests.get(
            f"{API}/{endpoint_id}/status/{job_id}", headers=HEADERS, timeout=60
        ).json().get("status")
        if status in {"COMPLETED", "FAILED", "CANCELLED"}:
            break
        time.sleep(poll_interval)


def chat_stream(messages, max_tokens=256, **sampling):
    """token-by-token llm output, printed live; returns the final result."""
    print("streaming: ", end="", flush=True)
    final = {}
    for chunk in stream(
        os.environ["LLM_STREAM_ENDPOINT_ID"],
        {"messages": messages, "max_tokens": max_tokens, **sampling},
    ):
        if "delta" in chunk:
            print(chunk["delta"], end="", flush=True)
        else:
            final = chunk
    print()
    return final


def transcribe_stream(audio_bytes, language=None):
    """rolling partial transcript of a full recording; returns the final result."""
    payload = {"audio": base64.b64encode(audio_bytes).decode(), "stream": True}
    if language:
        payload["language"] = language
    final = {}
    for chunk in stream(os.environ["STT_STREAM_ENDPOINT_ID"], payload):
        if "partial" in chunk:
            print(f"\rpartial: {chunk['partial']}", end="", flush=True)
        else:
            final = chunk
    print()
    return final


print("ready")

### token-by-token chat (formatted view)

In [ ]:
reply = chat_stream(
    [{"role": "user", "content": "Explain speculative decoding in two sentences."}]
)
reply

### token-by-token chat (raw chunks)

In [ ]:
# raw view: every chunk as it lands, one per line - the clearest way to
# see that output really is streaming
for chunk in stream(
    os.environ["LLM_STREAM_ENDPOINT_ID"],
    {"messages": [{"role": "user", "content": "Count from 1 to 15."}], "max_tokens": 48},
):
    print(chunk, flush=True)

### live partial transcript of your voice (formatted view)

In [ ]:
my_voice = record(seconds=5)
display(Audio(my_voice, rate=16000))
transcribe_stream(my_voice)

### live partial transcript (raw chunks)

In [ ]:
# raw view for stt partials: one line per 2s decode chunk
for chunk in stream(
    os.environ["STT_STREAM_ENDPOINT_ID"],
    {"audio": base64.b64encode(record(seconds=6)).decode(), "stream": True},
):
    print(chunk, flush=True)